# Código maestro — clasificación de emociones faciales mediante CNN

**Tesis:** *Modelos de redes neuronales convolucionales (CNNs) para determinar las emociones faciales en los postulantes a la Universidad Nacional del Altiplano Puno, 2024.*

**Este único cuaderno contiene el procedimiento completo:** control de calidad y preprocesamiento (MTCNN + alineación + CLAHE), sugerencia de etiquetas (DeepFace + ViT), separación de datos, búsqueda bayesiana, entrenamiento de cinco arquitecturas, ajuste fino, evaluación, análisis de cinco subconjuntos, validación cruzada REAL si se ejecuta, Grad-CAM y robustez visual.

**Integridad científica:** no existen cifras de resultados prefijadas. Todas las métricas y gráficos proceden de la ejecución. Se conservan las diferencias explícitas entre las técnicas observadas en el código recibido y las mejoras metodológicas. Este cuaderno reconstruido no certifica por sí mismo la ejecución histórica de 2024. En particular, cinco evaluaciones sobre el test **NO** equivalen a entrenamiento 5-fold.

**Privacidad:** nunca publiques las imágenes faciales, los IDs reversibles, rutas privadas, las predicciones de personas o el cuaderno con resultados que incluyan rostros. Los datos se mantienen en Google Drive privado.

**Uso:** ejecuta las celdas en orden y configura las rutas y opciones de la sección 1. La primera ejecución completa requiere tiempo de GPU y los datos/etiquetas reales. No avances al entrenamiento si las etiquetas no están definidas o no se han auditado.


## 0. Preparar entorno de Colab
Active **Entorno de ejecución → Cambiar tipo de entorno → GPU**. Instale dependencias antes de ejecutar los módulos. DeepFace y Transformers se utilizan exclusivamente si se activa la fase de pseudoetiquetado.

In [ ]:
# Celda de instalación para Google Colab.
%pip -q install "keras-tuner>=1.4,<2" "mtcnn>=1.0,<2" "opencv-python-headless>=4.8" "seaborn>=0.13" "pandas>=2.0" "scikit-learn>=1.3"
# Si se activa la etapa de autoetiquetado, ejecutar adicionalmente en una celda:
# %pip -q install "deepface>=0.0.93" "tf-keras" "transformers>=4.40,<5" "torch"

## 1. Configuración única del experimento
Indica las rutas reales. Si ya tienes imágenes **preprocesadas y etiquetadas**, utiliza directamente un manifiesto CSV: `image_id,image_path,subject_id,emotion_label,label_source`. Los `subject_id` deben ser seudónimos estables, no DNIs ni nombres. Para etiquetas verificadas por evaluadores, `label_source=human_verified`; para etiquetas del ensamble automático, `label_source=ensemble_automatico_no_verificado`.

In [ ]:
from pathlib import Path
import os, json, platform, random, hashlib
import numpy as np
import pandas as pd
from IPython.display import display

# Los datos personales deben permanecer en la unidad privada.
from google.colab import drive
drive.mount('/content/drive')

# CAMBIAR ESTAS RUTAS ANTES DE EJECUTAR:
CARPETA_TRABAJO = Path('/content/drive/MyDrive/CNN_UNA_EXPERIMENTO_PRIVADO')
CSV_CAPTURAS_ORIGINALES = ''  # opcional, CSV: image_id,image_path,subject_id
CSV_MANIFIESTO_ETIQUETADO = ''  # CSV privado con etiquetas válidas, generado o aportado por la investigación
CARPETA_RESULTADOS = CARPETA_TRABAJO / 'resultados_ejecucion'

# Etapas de preparación; activa únicamente aquellas que vas a realizar realmente.
EJECUTAR_CURACION_MTCNN = False
EJECUTAR_SUGERENCIA_ETIQUETAS_DEEPFACE_VIT = False
EJECUTAR_USO_EXPLICITO_DE_PSEUDOETIQUETAS = False

# Experimento. La optimización produce parámetros por cada arquitectura.
EJECUTAR_OPTIMIZACION_BAYESIANA = True
EJECUTAR_ENTRENAMIENTO_5_CNN = True
EJECUTAR_VALIDACION_CRUZADA_REAL_5FOLD = False  # costosa: 25 entrenamientos adicionales, NO realizada en código histórico
EJECUTAR_EVALUACION_TEST = True
EJECUTAR_PRUEBAS_ROBUSTEZ = True
EJECUTAR_GRADCAM_PRIVADO = False  # genera imágenes faciales: solo uso privado y autorizado

# False por defecto: las pseudoetiquetas NO equivalen a verdad de terreno.
PERMITIR_PSEUDOETIQUETAS = False

# Opciones fieles a las etapas observadas en el código recibido (sin cifras finales prefijadas).
SEMILLA = 42
NUMERO_ENSAYOS_BAYESIANOS = 3
NUMERO_EPOCAS_TUNER = 5
FRACCION_TEST = 0.20

CARPETA_TRABAJO.mkdir(parents=True,exist_ok=True)
CARPETA_RESULTADOS.mkdir(parents=True,exist_ok=True)
print('Directorio PRIVADO de trabajo:',CARPETA_TRABAJO)


## 2. Funciones de configuración y control de calidad
Los parámetros del entrenamiento se mantienen visibles y se registran para auditoría. No hay métricas escritas previamente.

In [ ]:
"""Configuración declarativa y trazabilidad de experimentos."""
from dataclasses import dataclass
from pathlib import Path
import json
import random
import os

EMOTIONS = ('tristeza', 'alegría', 'ira', 'miedo', 'sorpresa', 'disgusto')
ARCHITECTURES = ('Baseline', 'ResNet50', 'EfficientNetB0', 'DenseNet121', 'VGG19')
# Tabla de optimización de la tesis: referencia a contrastar, NO valores obtenidos aquí.

@dataclass(frozen=True)
class Settings:
    seed: int = 42
    image_size: int = 224
    batch_size: int = 32  # Batch de 32 registrado en el código recibido; ajustar por VRAM y documentarlo
    head_epochs: int = 20
    finetune_epochs: int = 30
    head_patience: int = 3
    finetune_patience: int = 4
    reduce_lr_on_plateau: bool = False
    use_class_weights: bool = False
    label_smoothing: float = 0.0
    fine_tune_conv_layers: int = 4
    fine_tune_lr: float = 1e-5
    workers: int = 1


def load_settings(config_path: str | None = None) -> Settings:
    if not config_path:
        return Settings()
    import yaml
    obj = yaml.safe_load(Path(config_path).read_text(encoding='utf-8')) or {}
    unknown = set(obj) - set(Settings.__dataclass_fields__)
    if unknown:
        raise ValueError(f'Configuración desconocida: {sorted(unknown)}')
    result = Settings(**obj)
    if result.image_size != 224 or result.batch_size < 1 or result.label_smoothing < 0:
        raise ValueError('Se requieren imágenes 224x224, batch positivo y smoothing no negativo.')
    return result


def fix_seed(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    try:
        import numpy as np
        np.random.seed(seed)
        import tensorflow as tf
        tf.keras.utils.set_random_seed(seed)
        try:
            tf.config.experimental.enable_op_determinism()
        except Exception:
            pass
    except ImportError:
        pass


def write_environment(folder: str | Path, cfg: Settings, extra: dict | None = None):
    import platform
    from dataclasses import asdict
    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)
    versions = {'python': platform.python_version()}
    for name in ('numpy', 'pandas', 'sklearn', 'tensorflow', 'keras', 'cv2', 'mtcnn'):
        try:
            module = __import__(name)
            versions[name] = str(getattr(module, '__version__', 'instalado'))
        except (ImportError, RuntimeError):
            versions[name] = 'no disponible'
    meta = {'settings': asdict(cfg), 'versions': versions,
            'historical_execution_verified': False, 'reported_results_are_not_training_targets': True,
            'notes': 'Implementación reconstruida: no certifica la ejecución histórica. Los resultados dependen de datos y pesos reales.'}
    meta.update(extra or {})
    (folder / 'ejecucion.json').write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')

config=Settings(seed=SEMILLA, batch_size=32, head_epochs=20, finetune_epochs=30, fine_tune_conv_layers=4, fine_tune_lr=1e-5, label_smoothing=0.0, reduce_lr_on_plateau=False)
print(config)


In [ ]:
"""Control de calidad MTCNN, desenfoque, duplicación y alineación facial."""
from pathlib import Path
import hashlib
import math
import pandas as pd


def detect_and_align(image_bgr, detector, *, min_pixels=100, confidence_min=0.70,
                     max_eye_tilt_degrees=45, min_laplacian_var=60, target_size=224):
    import cv2
    import numpy as np
    if image_bgr is None or min(image_bgr.shape[:2]) < min_pixels:
        return None,'lectura_fallida_o_baja_resolucion',{}
    blur = float(cv2.Laplacian(cv2.cvtColor(image_bgr,cv2.COLOR_BGR2GRAY),cv2.CV_64F).var())
    if blur < min_laplacian_var:
        return None,'desenfoque',{'nitidez_laplaciana':blur}
    rgb=cv2.cvtColor(image_bgr,cv2.COLOR_BGR2RGB)
    faces=detector.detect_faces(rgb)
    if not faces:
        return None,'rostro_no_detectado',{'nitidez_laplaciana':blur}
    face=max(faces,key=lambda f:float(f.get('confidence',0)))
    confidence=float(face.get('confidence',0))
    if confidence < confidence_min:
        return None,'baja_confianza_mtcnn',{'confianza_mtcnn':confidence}
    kp=face['keypoints']
    l=np.array(kp['left_eye'],dtype=float); r=np.array(kp['right_eye'],dtype=float)
    angle=math.degrees(math.atan2(r[1]-l[1],r[0]-l[0]))
    if abs(angle)>max_eye_tilt_degrees:
        return None,'rotacion_excesiva',{'angulo_ojos':round(angle,2)}
    h,w=image_bgr.shape[:2]
    center=tuple(((l+r)/2).tolist())
    M=cv2.getRotationMatrix2D(center,angle,1.0)
    rotated=cv2.warpAffine(image_bgr,M,(w,h),flags=cv2.INTER_LINEAR,borderMode=cv2.BORDER_REFLECT)
    # Transformar las cuatro esquinas del bounding box original antes de recortar.
    x,y,bw,bh=face['box']
    corners=np.array([[x,y,1],[x+bw,y,1],[x,y+bh,1],[x+bw,y+bh,1]],dtype=float)
    tc=corners @ M.T
    x1,y1=tc.min(axis=0); x2,y2=tc.max(axis=0)
    pad_x=.1*(x2-x1); pad_y=.1*(y2-y1)
    x1=max(0,int(x1-pad_x));x2=min(w,int(x2+pad_x))
    y1=max(0,int(y1-pad_y));y2=min(h,int(y2+pad_y))
    if x2<=x1 or y2<=y1:
        return None,'recorte_invalido',{}
    crop=rotated[y1:y2,x1:x2]
    gray=cv2.cvtColor(crop,cv2.COLOR_BGR2GRAY)
    clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8)).apply(gray)
    resized=cv2.resize(clahe,(target_size,target_size),interpolation=cv2.INTER_AREA)
    out=cv2.cvtColor(resized,cv2.COLOR_GRAY2BGR)
    return out,'aceptada',{'confianza_mtcnn':round(confidence,5),'angulo_ojos':round(angle,2),'nitidez_laplaciana':round(blur,2)}


def curate(input_csv: str, output_dir: str, *, max_images: int | None = None):
    """Acepta CSV de capturas sin etiquetas; el ensamble se aplica DESPUÉS de curar."""
    import cv2
    from mtcnn import MTCNN
    output_dir=Path(output_dir).resolve(); output_dir.mkdir(parents=True,exist_ok=True)
    images=output_dir/'private_curated_images';images.mkdir(exist_ok=True)
    input_path=Path(input_csv).resolve()
    rows=pd.read_csv(input_path,dtype=str).fillna('')
    required={'image_id','image_path','subject_id'}
    if not required <= set(rows):
        raise ValueError(f'El CSV de origen requiere {sorted(required)}')
    if rows['image_id'].duplicated().any():
        raise ValueError('Hay IDs duplicados en el CSV de origen.')
    detector=MTCNN()
    seen=set(); accepted=[]; log=[]
    for j, r in rows.iterrows():
        if max_images is not None and j>=max_images:
            break
        src=Path(r['image_path']);src=(input_path.parent/src).resolve() if not src.is_absolute() else src
        meta={'image_id':r['image_id']}
        if not src.is_file():
            log.append({**meta,'status':'rechazada','reason':'archivo_no_existe'});continue
        fingerprint=hashlib.sha256(src.read_bytes()).hexdigest()
        if fingerprint in seen:
            log.append({**meta,'status':'rechazada','reason':'duplicado_exacto'});continue
        seen.add(fingerprint)
        img=cv2.imread(str(src))
        crop,reason,details=detect_and_align(img,detector)
        log.append({**meta,'status':'aceptada' if crop is not None else 'rechazada','reason':reason,**details})
        if crop is None:continue
        safe_id=hashlib.sha256(r['image_id'].encode()).hexdigest()[:20]
        filename=f'{safe_id}.png';dst=images/filename
        cv2.imwrite(str(dst),crop)
        accepted.append({'image_id':safe_id,'image_path':f'private_curated_images/{filename}',
                         'subject_id':r['subject_id'],'emotion_label':r.get('emotion_label',''),
                         'label_source':r.get('label_source','pendiente'),'origin':'original',
                         'sha256':hashlib.sha256(dst.read_bytes()).hexdigest()})
    pd.DataFrame(log).to_csv(output_dir/'control_calidad_privado.csv',index=False)
    pd.DataFrame(accepted).to_csv(output_dir/'manifest_curado_privado.csv',index=False)
    return len(accepted),len(log)


## 3. Etiquetado y control de las seis emociones
El ensamble DeepFace + ViT genera **sugerencias**; si decides usarlas directamente, el resultado solo mide concordancia con pseudoetiquetas. Una etiqueta verificada por una persona debe estar efectivamente revisada. La clase `neutral` no se asigna a otra emoción por obligación.

In [ ]:
"""Sugerencias de pseudoetiquetado DeepFace+ViT, NO verdades de terreno.

No reasigna neutral a las seis emociones. Obliga a revisión humana antes de
tratar las etiquetas como verificación de la tesis.
"""
from pathlib import Path
import pandas as pd

ALIASES={
 'happy':'alegría','happiness':'alegría','joy':'alegría','sad':'tristeza',
 'sadness':'tristeza','angry':'ira','anger':'ira','fear':'miedo','fearful':'miedo',
 'surprise':'sorpresa','surprised':'sorpresa','disgust':'disgusto',
 'disgusted':'disgusto','neutral':'neutral'
}


def normalize_scores(raw:dict)->dict:
    scores={e:0.0 for e in ('alegría','tristeza','ira','miedo','sorpresa','disgusto','neutral')}
    for label,v in raw.items():
        canonical=ALIASES.get(str(label).strip().lower())
        if canonical is not None:scores[canonical]+=float(v)
    s=sum(scores.values())
    if s<=0:raise ValueError('El modelo de etiquetado no devolvió probabilidades utilizables.')
    return {k:v/s for k,v in scores.items()}


def consensus_scores(deepface:dict,vit:dict,*,neutral_limit:float=.20,confidence_min:float=.60):
    a=normalize_scores(deepface);b=normalize_scores(vit)
    combined={k:(a[k]+b[k])/2 for k in a}
    six={k:v for k,v in combined.items() if k!='neutral'}
    best=max(six,key=six.get)
    agree=max((k for k in a if k!='neutral'),key=a.get)==max((k for k in b if k!='neutral'),key=b.get)
    approved_candidate=agree and combined['neutral']<=neutral_limit and six[best]>=confidence_min
    return {'suggested_label':best if approved_candidate else '',
            'provisional_top_label':best,'confidence':round(six[best],5),
            'neutral_probability':round(combined['neutral'],5),
            'model_agreement':bool(agree),
            'status':'requires_human_review',
            'reason':'sugerencia_probable' if approved_candidate else 'neutral_o_discrepancia'}


def suggest_labels(curated_csv:str, output_csv:str):
    """Lanza modelos solo sobre archivos locales y devuelve sugerencias NO verificadas."""
    import cv2
    from transformers import pipeline
    from deepface import DeepFace
    import numpy as np
    import tempfile
    source=Path(curated_csv).resolve();rows=pd.read_csv(source,dtype=str).fillna('')
    vit=pipeline('image-classification',model='dima806/facial_emotions_image_detection')
    out=[]
    for _,row in rows.iterrows():
        filepath=Path(row['image_path'])
        filepath=(source.parent/filepath).resolve() if not filepath.is_absolute() else filepath
        if not filepath.is_file():
            out.append({'image_id':row['image_id'],'status':'error','reason':'file_not_found'});continue
        try:
            with tempfile.NamedTemporaryFile(suffix='.png',delete=False) as tmp:
                temp_path=Path(tmp.name)
            try:
                img=cv2.imread(str(filepath));cv2.imwrite(str(temp_path),cv2.resize(img,(48,48)))
                df_results=DeepFace.analyze(img_path=str(temp_path),actions=['emotion'],enforce_detection=False,detector_backend='skip')
                if isinstance(df_results,list):df_results=df_results[0]
                a={k:float(v)/100 for k,v in df_results['emotion'].items()}
                b={r['label']:float(r['score']) for r in vit(str(filepath),top_k=None)}
                candidate=consensus_scores(a,b)
                out.append({'image_id':row['image_id'],**candidate})
            finally:
                temp_path.unlink(missing_ok=True)
        except Exception as exc:
            out.append({'image_id':row['image_id'],'status':'error','reason':type(exc).__name__})
    pd.DataFrame(out).to_csv(output_csv,index=False)
    print('Todas las salidas son sugerencias que REQUIEREN revisión; no sobrescribir automáticamente emotion_label.')
    return len(out)


In [ ]:
"""Integra propuestas DeepFace+ViT sin confundirlas con etiquetas humanas."""
from pathlib import Path
import pandas as pd



def assemble_suggestions(curated_csv, suggestions_csv, output_csv, *, allow_automatic_labels=False):
    if not allow_automatic_labels:
        raise ValueError('Para usar etiquetas automáticas indique --allow-automatic-labels. La evaluación medirá concordancia con pseudoetiquetas.')
    source=Path(curated_csv).resolve(); suggestions=Path(suggestions_csv).resolve()
    x=pd.read_csv(source,dtype=str).fillna(''); y=pd.read_csv(suggestions,dtype=str).fillna('')
    for field in ('image_id','suggested_label'):
        if field not in y:raise ValueError(f'Falta {field} en suggestions')
    if y['image_id'].duplicated().any():raise ValueError('image_id duplicados en sugerencias')
    if x['image_id'].duplicated().any():raise ValueError('image_id duplicados en curación')
    x=x.drop(columns=['emotion_label','label_source'],errors='ignore')
    out=x.merge(y[['image_id','suggested_label']],on='image_id',how='left',validate='one_to_one')
    valid=out['suggested_label'].isin(EMOTIONS)
    omitted=int((~valid).sum());out=out.loc[valid].copy()
    if out.empty:raise ValueError('Ninguna pseudoetiqueta aprobada por los filtros: revisar neutral, umbrales y registros.')
    out['emotion_label']=out.pop('suggested_label')
    out['label_source']='ensemble_automatico_no_verificado'
    output=Path(output_csv).resolve();output.parent.mkdir(parents=True,exist_ok=True)
    # Reubicar rutas relativas respecto a la nueva ubicación del archivo.
    import os
    out['image_path']=[os.path.relpath((source.parent/p).resolve() if not Path(p).is_absolute() else Path(p),output.parent)
                       for p in out['image_path']]
    out.to_csv(output,index=False)
    return {'etiquetas_automaticas':len(out),'sin_consenso_suficiente':omitted,
            'archivo':str(output),'advertencia':'Son etiquetas automáticas. No equivalen a emociones reales verificadas independientemente.'}


## 4. Manifiesto, separación 80/20 y controles contra fuga de información
Primero se hace la división de imágenes originales o curadas; jamás se aumenta el conjunto de test. Se comprueban archivos duplicados, similitudes visuales y sujetos repetidos. El resultado depende del CSV realmente suministrado.

In [ ]:
"""Validación y particiones sin fuga de información entre personas/derivados."""
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd


MANDATORY = ('image_id', 'image_path', 'emotion_label', 'subject_id', 'label_source')


def visual_dhash(image_file: str) -> str:
    """Huella visual simple de 64 bits; solo para detectar candidatos a cuasi-duplicado."""
    import cv2
    img = cv2.imread(image_file, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError('Archivo ilegible al calcular dHash.')
    small = cv2.resize(img, (9, 8), interpolation=cv2.INTER_AREA)
    bits = (small[:, 1:] > small[:, :-1]).flatten()
    number = sum((int(bit) << i) for i, bit in enumerate(bits))
    return f'{number:016x}'



def load_manifest(file_path: str | Path, *, allow_pseudo: bool = False, check_files: bool = True) -> pd.DataFrame:
    path = Path(file_path).expanduser().resolve()
    df = pd.read_csv(path, dtype=str).fillna('')
    missing = set(MANDATORY) - set(df.columns)
    if missing:
        raise ValueError(f'Faltan columnas obligatorias: {sorted(missing)}. Revisar examples/manifest_ejemplo.csv')
    if df.empty:
        raise ValueError('El manifiesto está vacío.')
    for col in MANDATORY:
        if (df[col].astype(str).str.strip() == '').any():
            raise ValueError(f'Hay valores vacíos en {col}. Para identidad use ID pseudonimizado, NO nombre/DNI.')
    bad = set(df['emotion_label']) - set(EMOTIONS)
    if bad:
        raise ValueError(f'Etiquetas desconocidas: {sorted(bad)}; orden esperado {EMOTIONS}.')
    if df['image_id'].duplicated().any():
        raise ValueError('image_id duplicado: verifique archivos repetidos.')
    if 'origin' in df and (df['origin'].str.lower() != 'original').any():
        raise ValueError('El manifiesto experimental debe contener SOLO imágenes originales; aumentos exclusivamente en memoria sobre train.')
    if df['image_path'].map(lambda p: Path(p).name.lower().startswith('aug_')).any():
        raise ValueError('Se detectaron archivos aug_* potencialmente sintéticos. El dataset de evaluación debe incluir solo originales.')
    if not allow_pseudo and (df['label_source'] != 'human_verified').any():
        raise ValueError('Hay etiquetas no verificadas por personas. Se requiere revisión humana o --allow-pseudo con resultados interpretados solo como concordancia con pseudoetiquetas.')
    df['resolved_path'] = df['image_path'].map(lambda p: str((path.parent / p).resolve()) if not Path(p).is_absolute() else str(Path(p).resolve()))
    if check_files:
        no_exist = df.loc[~df['resolved_path'].map(lambda p: Path(p).is_file()), 'image_id'].tolist()
        if no_exist:
            raise FileNotFoundError(f'No existen {len(no_exist)} imágenes; ejemplos de IDs: {no_exist[:5]}')
        # Comprobar hashes reales, sin confiar en un CSV de metadatos potencialmente obsoleto.
        df['sha256'] = df['resolved_path'].map(lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest())
        if df['sha256'].duplicated().any():
            raise ValueError('El manifiesto contiene archivos idénticos por SHA256: retirar duplicados ANTES de separar train/test.')
        df['dhash'] = df['resolved_path'].map(visual_dhash)
    return df


def ensure_no_leak(train: pd.DataFrame, test: pd.DataFrame):
    for field in ('image_id', 'subject_id'):
        intersection = set(train[field]) & set(test[field])
        if intersection:
            raise ValueError(f'Fuga entre train/test: {len(intersection)} valores de {field} repetidos; primeros {sorted(intersection)[:3]}')
    if 'sha256' in train and 'sha256' in test:
        a = set(train['sha256']) - {''}; b = set(test['sha256']) - {''}
        if a & b:
            raise ValueError('Fuga de imágenes idénticas (SHA256) entre train/test.')
    if 'dhash' in train and 'dhash' in test:
        # No se declara que los dHash sean identidad personal; se reporta posible fuga visual.
        ta = [int(s,16) for s in train['dhash']]
        tb = [int(s,16) for s in test['dhash']]
        if any((a ^ b).bit_count() <= 2 for a in ta for b in tb):
            raise ValueError('Posible fuga por casi duplicados visuales dHash (distancia <=2) entre train/test; revisar manualmente.')
    if 'parent_id' in train and 'parent_id' in test:
        if (set(train['parent_id']) - {''}) & (set(test['parent_id']) - {''}):
            raise ValueError('Imagen original y sus derivadas aparecen a ambos lados de la partición.')


def split_train_test(df: pd.DataFrame, *, seed: int = 42, test_fraction: float = 0.2) -> pd.DataFrame:
    from sklearn.model_selection import StratifiedShuffleSplit, GroupShuffleSplit
    if not 0 < test_fraction < 1:
        raise ValueError('test_fraction debe estar en (0,1).')
    if df['emotion_label'].value_counts().min() < 2:
        raise ValueError('Cada emoción requiere al menos 2 imágenes para la partición estratificada.')
    # GroupShuffleSplit previene compartir una misma persona en train/test.
    # Si cada imagen es una persona distinta, StratifiedShuffleSplit conserva proporciones con mayor precisión.
    if not df['subject_id'].duplicated().any():
        chooser = StratifiedShuffleSplit(n_splits=1, test_size=test_fraction, random_state=seed)
        train_ix, test_ix = next(chooser.split(df, df['emotion_label']))
        method = 'StratifiedShuffleSplit (una imagen por sujeto)'
    else:
        chooser = GroupShuffleSplit(n_splits=200, test_size=test_fraction, random_state=seed)
        pairs = chooser.split(df, df['emotion_label'], groups=df['subject_id'])
        target = df['emotion_label'].value_counts(normalize=True).reindex(EMOTIONS, fill_value=0)
        def score(pair):
            a,b=pair
            if len(set(df.iloc[a]['emotion_label'])) < len(EMOTIONS) or len(set(df.iloc[b]['emotion_label'])) < len(EMOTIONS):
                return np.inf
            dist=df.iloc[b]['emotion_label'].value_counts(normalize=True).reindex(EMOTIONS,fill_value=0)
            return abs(len(b)/len(df)-test_fraction)*5 + float(abs(dist-target).sum())
        best=min(pairs,key=score)
        if not np.isfinite(score(best)):
            raise ValueError('No se pudo construir partición por sujeto con las seis emociones en ambos grupos.')
        train_ix,test_ix=best
        method='GroupShuffleSplit (sujeto independiente, selección por balance)'
    out=df.copy()
    out['split']=''
    out.loc[df.index[train_ix],'split']='train'
    out.loc[df.index[test_ix],'split']='test'
    ensure_no_leak(out[out['split']=='train'],out[out['split']=='test'])
    out.attrs['split_method']=method
    return out


def check_split(df: pd.DataFrame):
    if set(df['split']) != {'train','test'}:
        raise ValueError('split debe ser train o test y contener ambos.')
    ensure_no_leak(df[df['split']=='train'], df[df['split']=='test'])
    return True


def summary(df: pd.DataFrame) -> pd.DataFrame:
    cols=['emotion_label','split'] if 'split' in df else ['emotion_label']
    return df.groupby(cols,dropna=False).size().rename('n_imagenes').reset_index()


def manifest_fingerprint(df: pd.DataFrame) -> str:
    columns=sorted(set(('image_id','emotion_label','subject_id','split')) & set(df))
    raw=df[columns].sort_values(columns).to_csv(index=False).encode()
    return hashlib.sha256(raw).hexdigest()


def assert_same_experiment_manifest(df: pd.DataFrame, results_dir: str | Path):
    """Impide atribuir a un dataset distinto los pesos guardados en el mismo directorio."""
    folder=Path(results_dir)
    saved=folder/'huella_dataset.txt'
    if not saved.is_file():
        raise FileNotFoundError(f'No existe {saved}; se necesita entrenar primero con este manifiesto.')
    actual=manifest_fingerprint(df)
    historical=saved.read_text(encoding='utf-8').strip()
    if actual != historical:
        raise ValueError('Los modelos guardados corresponden a OTRO manifiesto de datos/splits. No se mezclan experimentos.')
    return True


## 5. Lectura, aumento solo en entrenamiento y preprocesamiento por arquitectura

In [ ]:
"""Lectura de imágenes y data augmentation EXCLUSIVAMENTE del entrenamiento."""



def get_preprocessing(architecture: str):
    import tensorflow as tf
    if architecture == 'Baseline':
        return lambda image: tf.cast(image,tf.float32)/255.0
    if architecture == 'ResNet50':
        return tf.keras.applications.resnet.preprocess_input
    if architecture == 'EfficientNetB0':
        return tf.keras.applications.efficientnet.preprocess_input  # identidad; Rescaling dentro de EfficientNet
    if architecture == 'DenseNet121':
        return tf.keras.applications.densenet.preprocess_input
    if architecture == 'VGG19':
        return tf.keras.applications.vgg19.preprocess_input
    raise ValueError(f'Arquitectura desconocida {architecture}')


def make_dataset(df, architecture: str, *, batch_size: int=16, seed: int=42,
                 augment: bool=False, shuffle: bool=False):
    import tensorflow as tf
    df=df.reset_index(drop=True)
    paths=df['resolved_path'].astype(str).tolist()
    y=[EMOTIONS.index(a) for a in df['emotion_label']]
    base=tf.data.Dataset.from_tensor_slices((paths,y))
    if shuffle:
        base=base.shuffle(len(paths),seed=seed,reshuffle_each_iteration=True)
    def read_image(path,label):
        image=tf.io.decode_image(tf.io.read_file(path),channels=3,expand_animations=False)
        image.set_shape((None,None,3))
        image=tf.image.resize(image,(224,224),method='bilinear')
        label=tf.one_hot(label,depth=6)
        return tf.cast(image,tf.float32),tf.cast(label,tf.float32)
    base=base.map(read_image,num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size)
    augmentation=tf.keras.Sequential([
        tf.keras.layers.RandomFlip('horizontal',seed=seed),
        # Aproximadamente +/- 15 grados (factor es fracción de vuelta completa).
        tf.keras.layers.RandomRotation(15.0/360.0,fill_mode='reflect',seed=seed+1),
        tf.keras.layers.RandomZoom(height_factor=(-0.20,0.20),width_factor=(-0.20,0.20),seed=seed+2)
    ])
    preproc=get_preprocessing(architecture)
    def transform(images,labels):
        if augment:
            images=augmentation(images,training=True)
        return preproc(tf.cast(images,tf.float32)), labels
    base=base.map(transform,num_parallel_calls=1).prefetch(tf.data.AUTOTUNE)
    opt=tf.data.Options();opt.experimental_deterministic=True
    return base.with_options(opt)


def predict_labels(model, df, arch: str, batch_size: int=16):
    import numpy as np
    ds=make_dataset(df,arch,batch_size=batch_size,augment=False,shuffle=False)
    probabilities=model.predict(ds,verbose=0)
    targets=np.array([EMOTIONS.index(a) for a in df['emotion_label']])
    pred=np.argmax(probabilities,axis=1)
    return targets,pred,probabilities


## 6. Arquitecturas y entrenamiento
Baseline, ResNet50, EfficientNetB0, DenseNet121 y VGG19. Búsqueda bayesiana en el subconjunto de entrenamiento, una etapa de cabeza congelada y ajuste fino sobre las últimas capas convolucionales. Se reserva el test hasta la evaluación final.

In [ ]:
"""Cinco arquitecturas, misma cabeza densa y preprocesamiento por familia.

La red recibe tensores ya preprocesados por `tf_dataset.py`. La normalización
no se ejecuta dos veces; EfficientNet conserva el preprocesamiento integrado.
"""



def build_model(arch: str, *, dense_units: int, dropout_rate: float,
                learning_rate: float, label_smoothing: float,
                pretrained: bool = True):
    import tensorflow as tf
    if arch not in ARCHITECTURES:
        raise ValueError(f'Arquitectura no reconocida {arch}')
    layers=tf.keras.layers
    image_input=layers.Input(shape=(224,224,3),name='image_preprocessed')
    if arch == 'Baseline':
        x=image_input
        for i, filters in enumerate((32,64,128)):
            x=layers.Conv2D(filters,3,padding='same',activation='relu',name=f'conv_{i+1}')(x)
            x=layers.BatchNormalization(name=f'bn_{i+1}')(x)
            x=layers.MaxPooling2D(2,name=f'pool_{i+1}')(x)
        backbone=tf.keras.Model(image_input,x,name='backbone')
    else:
        factory={
            'ResNet50':tf.keras.applications.ResNet50,
            'EfficientNetB0':tf.keras.applications.EfficientNetB0,
            'DenseNet121':tf.keras.applications.DenseNet121,
            'VGG19':tf.keras.applications.VGG19
        }[arch]
        core=factory(weights='imagenet' if pretrained else None,
                     include_top=False,input_shape=(224,224,3))
        backbone=tf.keras.Model(core.input,core.output,name='backbone')
        backbone.trainable=False
    head_input=layers.Input(shape=tuple(backbone.output_shape[1:]),name='feature_maps')
    # Baseline conserva Flatten como en el código inicial; los modelos transferidos usan GAP.
    h=layers.Flatten(name='flatten')(head_input) if arch=='Baseline' else layers.GlobalAveragePooling2D(name='gap')(head_input)
    h=layers.Dense(dense_units,activation='relu',name='dense_head')(h)
    h=layers.Dropout(dropout_rate,name='dropout_head')(h)
    result=layers.Dense(6,activation='softmax',name='emotion_probabilities')(h)
    classifier=tf.keras.Model(head_input,result,name='classifier')
    whole=tf.keras.Model(image_input,classifier(backbone(image_input)),name=f'cnn_{arch.lower()}')
    compile_model(whole,learning_rate,label_smoothing)
    return whole


def compile_model(model, lr:float, smoothing:float):
    import tensorflow as tf
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                  loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=smoothing),
                  metrics=[tf.keras.metrics.CategoricalAccuracy(name='accuracy')])


def unfreeze_last_conv(model, n_conv: int):
    import tensorflow as tf
    base=model.get_layer('backbone')
    base.trainable=True
    for l in base.layers:
        l.trainable=False
    conv=[l for l in base.layers if isinstance(l,tf.keras.layers.Conv2D)]
    if not conv:
        raise RuntimeError('Backbone no tiene Conv2D; comprobar arquitectura.')
    for l in conv[-n_conv:]:
        l.trainable=True
    # BatchNorm permanece congelado para evitar cambiar las estadísticas ImageNet.
    return [l.name for l in conv[-n_conv:]]


In [ ]:
"""Entrenamiento final, optimización bayesiana opcional y CV real de cinco folds."""
from pathlib import Path
import json
import numpy as np
import pandas as pd




def internal_train_val(df:pd.DataFrame,seed:int):
    internal=split_train_test(df.drop(columns=['split'],errors='ignore'),seed=seed,test_fraction=.15)
    a=internal[internal['split']=='train'].drop(columns='split').reset_index(drop=True)
    b=internal[internal['split']=='test'].drop(columns='split').reset_index(drop=True)
    return a,b


def _callbacks(save_path: Path, cfg: Settings, *, patience: int):
    import tensorflow as tf
    callbacks=[
      tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=patience,restore_best_weights=True),
      tf.keras.callbacks.ModelCheckpoint(str(save_path),monitor='val_loss',save_best_only=True,verbose=0)
    ]
    if cfg.reduce_lr_on_plateau:
        callbacks.insert(1, tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',factor=.5,patience=max(2,patience//2),min_lr=1e-7,verbose=1))
    return callbacks


def train_once(arch:str,train_df:pd.DataFrame,val_df:pd.DataFrame,
               folder: Path,cfg: Settings, hyperparameters:dict, *, pretrained:bool=True):
    import tensorflow as tf


    folder.mkdir(parents=True,exist_ok=True)
    fix_seed(cfg.seed)
    tf.keras.backend.clear_session()
    hp=hyperparameters[arch]
    model=build_model(arch,dense_units=int(hp['dense_units']),dropout_rate=float(hp['dropout_rate']),
                      learning_rate=float(hp['learning_rate']),label_smoothing=cfg.label_smoothing,
                      pretrained=pretrained)
    train_ds=make_dataset(train_df,arch,batch_size=cfg.batch_size,seed=cfg.seed,augment=True,shuffle=True)
    val_ds=make_dataset(val_df,arch,batch_size=cfg.batch_size,seed=cfg.seed,augment=False,shuffle=False)
    # Peso de clases estimado SOLO sobre train, para no filtrar información de validación/test.
    counts=train_df['emotion_label'].value_counts()
    class_weights={i:len(train_df)/(len(EMOTIONS)*counts.get(label,1)) for i,label in enumerate(EMOTIONS)} if cfg.use_class_weights else None
    first=folder/'etapa_A.keras'
    hist_A=model.fit(train_ds,validation_data=val_ds,epochs=cfg.head_epochs,
                     callbacks=_callbacks(first,cfg,patience=cfg.head_patience),verbose=2,class_weight=class_weights)
    a_min=float(min(hist_A.history['val_loss']))
    history={'etapa_A':hist_A.history,'etapa_B':None,'model':arch,
             'preprocess':arch,'class_weight':class_weights,'hyperparameters':hp,'configuracion':cfg.__dict__,
             'advertencia':'Reconstrucción corregida; no afirma ejecución histórica'}
    best_path=first
    if arch!='Baseline':
        conv_names=unfreeze_last_conv(model,cfg.fine_tune_conv_layers)
        compile_model(model,cfg.fine_tune_lr,cfg.label_smoothing)
        second=folder/'etapa_B.keras'
        hist_B=model.fit(train_ds,validation_data=val_ds,epochs=cfg.finetune_epochs,
                         callbacks=_callbacks(second,cfg,patience=cfg.finetune_patience),verbose=2,class_weight=class_weights)
        history['etapa_B']=hist_B.history
        history['capas_ajustadas']=conv_names
        b_min=float(min(hist_B.history['val_loss']))
        if b_min<a_min:
            best_path=second
    else:
        history['capas_ajustadas']=[]
    # No sobreescribir los pesos de mejor validación con los de la última época.
    model=tf.keras.models.load_model(best_path)
    final=folder/'modelo_final.keras'
    model.save(final)
    history['mejor_etapa']= 'B' if best_path!=first else 'A'
    history['mejor_val_loss']= min(a_min,float(min(history['etapa_B']['val_loss'])) if history['etapa_B'] else a_min)
    (folder/'historial.json').write_text(json.dumps(history,ensure_ascii=False,indent=2,default=float),encoding='utf-8')
    return model,history


def tune_architectures(df:pd.DataFrame, folder:str|Path,cfg:Settings, *, max_trials:int=3, trial_epochs:int=5):
    import tensorflow as tf
    import keras_tuner as kt


    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
    tr,val=internal_train_val(df[df['split']=='train'],cfg.seed)
    results={}
    for arch in ARCHITECTURES:
        tf.keras.backend.clear_session();fix_seed(cfg.seed)
        train_ds=make_dataset(tr,arch,batch_size=cfg.batch_size,seed=cfg.seed,augment=True,shuffle=True)
        val_ds=make_dataset(val,arch,batch_size=cfg.batch_size,seed=cfg.seed)
        def factory(hp):
            return build_model(arch,dense_units=hp.Choice('dense_units',[128,256,384,512]),
                               dropout_rate=hp.Choice('dropout_rate',[0.2,0.3,0.4,0.5]),
                               learning_rate=hp.Choice('learning_rate',[1e-3,1e-4]),
                               label_smoothing=cfg.label_smoothing)
        tuner=kt.BayesianOptimization(factory,objective='val_accuracy',max_trials=max_trials,
                                    executions_per_trial=1,directory=str(folder/'busqueda'),
                                    project_name=arch,overwrite=True,seed=cfg.seed)
        tuner.search(train_ds,validation_data=val_ds,epochs=trial_epochs,
                     callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=2)],verbose=2)
        hp=tuner.get_best_hyperparameters(1)[0]
        results[arch]={key:hp.get(key) for key in ('dense_units','dropout_rate','learning_rate')}
    (folder/'hiperparametros_optimizados.json').write_text(json.dumps(results,indent=2),encoding='utf-8')
    return results


def load_hyperparameters(optional_path:str|None):
    if optional_path is None:
        raise ValueError('No se proporcionó búsqueda bayesiana realizada. Este cuaderno no utilizará valores declarados en la tesis como sustituto de resultados de sintonización.')
    values=json.loads(Path(optional_path).read_text(encoding='utf-8'))
    for arch in ARCHITECTURES:
        for key in ('dense_units','dropout_rate','learning_rate'):
            if arch not in values or key not in values[arch]:
                raise ValueError(f'No se encontró {arch}/{key} en {optional_path}')
    return values


def train_final(df:pd.DataFrame,results_dir:str|Path,cfg:Settings,hps:dict, *, pretrained=True):
    import tensorflow as tf

    check_split(df)
    results_dir=Path(results_dir);results_dir.mkdir(parents=True,exist_ok=True)
    pool=df[df['split']=='train'].copy();tr,val=internal_train_val(pool,cfg.seed)
    (results_dir/'huella_dataset.txt').write_text(manifest_fingerprint(df)+'\n',encoding='utf-8')
    run_summary=[]
    for arch in ARCHITECTURES:
        print(f'\n=== Entrenamiento FINAL {arch} (test bloqueado) ===')
        model,hist=train_once(arch,tr,val,results_dir/'modelos'/arch,cfg,hps,pretrained=pretrained)
        run_summary.append({'Arquitectura':arch,'val_loss':hist['mejor_val_loss'],'etapa':hist['mejor_etapa']})
        del model;tf.keras.backend.clear_session()
    pd.DataFrame(run_summary).to_csv(results_dir/'validacion_interna.csv',index=False)
    return run_summary


def folds_train_only(df:pd.DataFrame,seed:int):
    from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
    pool=df[df['split']=='train'].reset_index(drop=True)
    if pool['subject_id'].duplicated().any():
        splitter=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=seed)
        iterator=splitter.split(pool,pool['emotion_label'],pool['subject_id'])
    else:
        splitter=StratifiedKFold(n_splits=5,shuffle=True,random_state=seed)
        iterator=splitter.split(pool,pool['emotion_label'])
    for fold,(tr,val) in enumerate(iterator,start=1):
        train_df=pool.iloc[tr].copy();val_df=pool.iloc[val].copy()
        ensure_no_leak(train_df,val_df)
        yield fold,train_df,val_df


def crossvalidate(df:pd.DataFrame,results_dir:str|Path,cfg:Settings,hps:dict, *,pretrained=True):
    import tensorflow as tf

    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    rows=[];root=Path(results_dir)
    for fold,tr,val in folds_train_only(df,cfg.seed):
        for arch in ARCHITECTURES:
            print(f'\n=== CV auténtica {arch} fold {fold}/5: entrenamiento desde cero ===')
            model,_=train_once(arch,tr,val,root/'cv'/arch/f'fold_{fold}',cfg,hps,pretrained=pretrained)
            true,pred,_=predict_labels(model,val,arch,cfg.batch_size)
            rows.append({'arquitectura':arch,'fold':fold,'n':len(val),
                         'accuracy':accuracy_score(true,pred),'precision_macro':precision_score(true,pred,average='macro',zero_division=0),
                         'recall_macro':recall_score(true,pred,average='macro',zero_division=0),
                         'f1_macro':f1_score(true,pred,average='macro',zero_division=0)})
            del model;tf.keras.backend.clear_session()
    frame=pd.DataFrame(rows);(root/'cv').mkdir(parents=True,exist_ok=True)
    frame.to_csv(root/'cv'/'resultados_5fold_reentrenados.csv',index=False)
    frame.groupby('arquitectura')[['accuracy','f1_macro']].agg(['mean','std']).to_csv(root/'cv'/'resumen_5fold.csv')
    return frame


## 7. Evaluación sin resultados prefijados
Los cinco subconjuntos de prueba reproducen el esquema de evaluación del texto recibido; se etiquetan correctamente como estabilidad del test, **no validación cruzada con reentrenamiento**. La validación cruzada auténtica está disponible como etapa adicional y se ejecuta por separado.

In [ ]:
"""Evaluación sobre TEST reservado: métricas por modelo y matrices verificables."""
from pathlib import Path
import json
import numpy as np
import pandas as pd



def metrics_dict(y_true,y_pred):
    from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix
    return {
       'accuracy':float(accuracy_score(y_true,y_pred)),
       'precision_macro':float(precision_score(y_true,y_pred,average='macro',labels=list(range(6)),zero_division=0)),
       'recall_macro':float(recall_score(y_true,y_pred,average='macro',labels=list(range(6)),zero_division=0)),
       'f1_macro':float(f1_score(y_true,y_pred,average='macro',labels=list(range(6)),zero_division=0)),
       'n':int(len(y_true)),
       'correctas':int((np.asarray(y_true)==np.asarray(y_pred)).sum()),
       'confusion_matrix':confusion_matrix(y_true,y_pred,labels=list(range(6))).tolist()
    }


def five_test_subsets(y_true,y_pred,seed=42):
    """Replica la *descripción* del protocolo histórico: cinco subconjuntos de test.
    NO es validación cruzada de entrenamiento. Los pesos permanecen fijos.
    """
    from sklearn.model_selection import KFold
    folds=KFold(n_splits=5,shuffle=True,random_state=seed)
    rows=[]
    for fold,(_,part) in enumerate(folds.split(y_true),start=1):
        score=metrics_dict(np.asarray(y_true)[part],np.asarray(y_pred)[part])
        rows.append({'fold_test':fold,'n_test':score['n'], 'accuracy':score['accuracy'],
                     'f1_macro':score['f1_macro'], 'correctas':score['correctas']})
    return rows


def plot_confusion(y_true,y_pred, output_path:Path, title:str):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from sklearn.metrics import ConfusionMatrixDisplay
    fig,ax=plt.subplots(figsize=(8,7))
    ConfusionMatrixDisplay.from_predictions(y_true,y_pred,labels=list(range(6)),
        display_labels=EMOTIONS,cmap='Blues',values_format='d',ax=ax,colorbar=False)
    ax.set_title(title)
    plt.setp(ax.get_xticklabels(),rotation=25,ha='right')
    fig.tight_layout();fig.savefig(output_path,dpi=220);plt.close(fig)


def plot_histories(results_dir:Path):
    """Curvas de todos los modelos, usando solamente registros guardados (no simulados)."""
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    for arch in ARCHITECTURES:
        path=results_dir/'modelos'/arch/'historial.json'
        if not path.exists():continue
        h=json.loads(path.read_text(encoding='utf-8'))
        for metric in ('accuracy','loss'):
            fig,ax=plt.subplots(figsize=(8,4.5));offset=0
            for phase in ('etapa_A','etapa_B'):
                history=h.get(phase)
                if history is None: continue
                vals=history[metric];val_vals=history['val_'+metric]
                epochs=np.arange(offset+1,offset+len(vals)+1)
                ax.plot(epochs,vals,label=f'{phase} entrenamiento')
                ax.plot(epochs,val_vals,label=f'{phase} validación',linestyle='--')
                offset+=len(vals)
            ax.set(xlabel='Época',ylabel=metric,title=f'{arch}: curvas reales de {metric}')
            ax.grid(alpha=.2);ax.legend();fig.tight_layout()
            fig.savefig(results_dir/'graficos'/f'curvas_{arch}_{metric}.png',dpi=200)
            plt.close(fig)


def evaluate_models(df,results_dir:str|Path,batch_size:int=16,seed:int=42):
    import tensorflow as tf

    from sklearn.metrics import classification_report

    folder=Path(results_dir)
    assert_same_experiment_manifest(df, folder)
    (folder/'graficos').mkdir(parents=True,exist_ok=True)
    (folder/'metricas').mkdir(parents=True,exist_ok=True)
    test=df[df['split']=='test'].reset_index(drop=True)
    if len(test)==0:raise ValueError('No existe conjunto de prueba.')
    summary=[]; historical=[]
    for arch in ARCHITECTURES:
        model_path=folder/'modelos'/arch/'modelo_final.keras'
        if not model_path.exists():raise FileNotFoundError(f'Falta modelo entrenado: {model_path}')
        model=tf.keras.models.load_model(model_path)
        actual,pred,prob=predict_labels(model,test,arch,batch_size)
        m=metrics_dict(actual,pred)
        out={'Modelo':arch,'Loss':float(model.evaluate(
            make_dataset(test,arch,batch_size=batch_size),verbose=0)[0]),
             'Accuracy (%)':round(m['accuracy']*100,2), 'Precision (Macro Avg)':round(m['precision_macro'],4),
             'Recall (Macro Avg)':round(m['recall_macro'],4), 'F1-Score (Macro Avg)':round(m['f1_macro'],4),
             'n_test':m['n'], 'n_correctas':m['correctas']}
        summary.append(out)
        report=classification_report(actual,pred,labels=list(range(6)),target_names=EMOTIONS,zero_division=0,output_dict=True)
        (folder/'metricas'/f'reporte_{arch}.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
        pd.DataFrame(m['confusion_matrix'],index=EMOTIONS,columns=EMOTIONS).to_csv(folder/'metricas'/f'confusion_{arch}.csv')
        plot_confusion(actual,pred,folder/'graficos'/f'confusion_{arch}.png',f'{arch} - Test independiente n={len(test)}')
        # NUNCA se publica la ruta completa ni fotos; image_id debe ser pseudónimo.
        per_image=pd.DataFrame({'image_id':test['image_id'], 'y_true': [EMOTIONS[i] for i in actual],
                                'y_pred':[EMOTIONS[i] for i in pred],
                                'confidence':np.max(prob,axis=1).round(6)})
        per_image.to_csv(folder/'metricas'/f'predicciones_PRIVADAS_{arch}.csv',index=False)
        for f in five_test_subsets(actual,pred,seed=seed):
            historical.append({'Modelo':arch,**f})
        del model;tf.keras.backend.clear_session()
    frame=pd.DataFrame(summary).sort_values('Accuracy (%)',ascending=False)
    frame.to_csv(folder/'metricas_finales_completas.csv',index=False)
    sub=pd.DataFrame(historical)
    sub.to_csv(folder/'estabilidad_5_subconjuntos_TEST_NO_CV.csv',index=False)
    sub.groupby('Modelo')['accuracy'].agg(['mean','std']).to_csv(folder/'estabilidad_5_subconjuntos_resumen.csv')
    plot_histories(folder)
    return frame


## 8. Robustez visual y Grad-CAM
Los escenarios de perturbación se aplican al test, sin usarlo en ningún ajuste de pesos. Los mapas Grad-CAM con rostros son privados.

In [ ]:
"""Cuatro perturbaciones del capítulo IV + una perturbación compuesta.

Todo se aplica sobre imágenes test originales, SIN entrenamiento, guardando sólo agregados.
No se confunde el estrés compuesto en validación con el estrés por escenario en test.
"""
from pathlib import Path
import math
import numpy as np
import pandas as pd




def perturb(rgb:np.ndarray,scenario:str,*,seed:int=42)->np.ndarray:
    import cv2
    image=rgb.astype(np.float32).copy()
    if scenario=='Control':
        pass
    elif scenario=='Gafas':
        image[50:95,40:184]=0
    elif scenario=='Mascarilla':
        image[115:210,35:190]=0
    elif scenario=='Sombras':
        fade=np.linspace(1.,.15,224,dtype=np.float32)[:,None,None]
        image=image*fade
    elif scenario=='Angulo':
        M=cv2.getRotationMatrix2D((112,112),12.0,1.0)
        image=cv2.warpAffine(image,M,(224,224),borderMode=cv2.BORDER_REFLECT)
    elif scenario=='RuidoCompuesto':
        rng=np.random.default_rng(seed)
        degrees=float(rng.uniform(-20,20));contrast=float(rng.uniform(.8,1.2))
        M=cv2.getRotationMatrix2D((112,112),degrees,1.0)
        image=cv2.warpAffine(image,M,(224,224),borderMode=cv2.BORDER_REFLECT)
        image=(image-127.5)*contrast+127.5
    else:raise ValueError(scenario)
    return np.clip(image,0,255).astype('float32')


def load_raw_images(df, *, size:int=224):
    import cv2
    images=[]
    for p in df['resolved_path']:
        raw=cv2.imread(p)
        if raw is None:raise ValueError('No se pudo cargar una imagen del manifiesto.')
        raw=cv2.cvtColor(cv2.resize(raw,(size,size)),cv2.COLOR_BGR2RGB)
        images.append(raw)
    return np.stack(images,axis=0).astype('float32')


def run_stress(df,results_dir:str|Path,batch_size:int=16,seed:int=42):
    import tensorflow as tf


    folder=Path(results_dir)
    assert_same_experiment_manifest(df, folder)
    test=df[df['split']=='test'].reset_index(drop=True)
    raw=load_raw_images(test);true=np.asarray([EMOTIONS.index(x) for x in test['emotion_label']])
    # Preparación UNA SOLA VEZ por escenario, para que todas las arquitecturas reciban el mismo estrés.
    scenarios=['Control','Gafas','Mascarilla','Sombras','Angulo','RuidoCompuesto']
    modified={scenario:np.stack([perturb(im,scenario,seed=seed+i) for i,im in enumerate(raw)]) for scenario in scenarios}
    rows=[]
    for arch in ARCHITECTURES:
        model=tf.keras.models.load_model(folder/'modelos'/arch/'modelo_final.keras')
        preprocess=get_preprocessing(arch)
        for scenario in scenarios:
            values=tf.convert_to_tensor(modified[scenario],dtype=tf.float32)
            model_input=preprocess(values)
            pred=np.argmax(model.predict(model_input,batch_size=batch_size,verbose=0),axis=1)
            row={'Modelo':arch,'Escenario':scenario,'n':len(true),
                 'Accuracy (%)':round(metrics_dict(true,pred)['accuracy']*100,2)}
            rows.append(row)
        del model;tf.keras.backend.clear_session()
    results=pd.DataFrame(rows)
    results.to_csv(folder/'metricas_estres_finales.csv',index=False)
    pivot=results.pivot(index='Modelo',columns='Escenario',values='Accuracy (%)').reset_index()
    pivot.to_csv(folder/'tabla_control_gafas_mascarilla_sombras_angulo.csv',index=False)
    return pivot


In [ ]:
"""Grad-CAM auténtico de la última capa Conv2D, no de la salida global del backbone."""
from pathlib import Path
import numpy as np



def grad_cam(model, image_preprocessed):
    import tensorflow as tf
    backbone=model.get_layer('backbone')
    head=model.get_layer('classifier')
    convs=[l for l in backbone.layers if isinstance(l,tf.keras.layers.Conv2D)]
    if not convs:raise RuntimeError('No se detectaron capas convolucionales para Grad-CAM.')
    last=convs[-1]
    extractor=tf.keras.Model(backbone.inputs,[last.output,backbone.output])
    image_preprocessed=tf.convert_to_tensor(image_preprocessed,dtype=tf.float32)
    with tf.GradientTape() as tape:
        conv_out,features=extractor(image_preprocessed,training=False)
        probs=head(features,training=False)
        target=tf.argmax(probs[0])
        class_output=probs[:,target]
    gradients=tape.gradient(class_output,conv_out)
    if gradients is None:raise RuntimeError('Grad-CAM no recibió gradientes; verificar arquitectura y capas.')
    pooled=tf.reduce_mean(gradients,axis=(0,1,2))
    heat=tf.einsum('hwc,c->hw',conv_out[0],pooled)
    heat=tf.nn.relu(heat)
    heat=heat/(tf.reduce_max(heat)+1e-8)
    return heat.numpy(),last.name,int(target),float(probs[0,target])


def export_gradcam(df,results_dir:str|Path, *, n_per_class:int=1,allow_private_export:bool=False):
    if not allow_private_export:
        raise PermissionError('Grad-CAM genera imágenes con rostros. Solo salida PRIVADA y autorización explícita: --allow-private-export.')
    import tensorflow as tf
    import cv2
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt



    folder=Path(results_dir)
    assert_same_experiment_manifest(df, folder)
    test=df[df['split']=='test'].copy()
    picks=test.groupby('emotion_label',sort=False).head(n_per_class).reset_index(drop=True)
    rgb=load_raw_images(picks)
    output=folder/'PRIVADO_gradcam_no_publicar';output.mkdir(parents=True,exist_ok=True)
    for arch in ARCHITECTURES:
        model=tf.keras.models.load_model(folder/'modelos'/arch/'modelo_final.keras')
        prep=get_preprocessing(arch)
        for k,frame in picks.iterrows():
            array=tf.convert_to_tensor(rgb[k:k+1],dtype=tf.float32)
            heat,layer,index,prob=grad_cam(model,prep(array))
            heat=cv2.resize(heat,(224,224))
            colored=cv2.applyColorMap(np.uint8(np.clip(heat,0,1)*255),cv2.COLORMAP_JET)
            colored=cv2.cvtColor(colored,cv2.COLOR_BGR2RGB)
            overlay=np.uint8(rgb[k]*.60+colored*.40)
            fig,ax=plt.subplots(1,2,figsize=(8,4))
            ax[0].imshow(rgb[k].astype(np.uint8));ax[0].set_title('Original (privado)')
            ax[1].imshow(overlay);ax[1].set_title(f'{arch}: {EMOTIONS[index]} ({prob:.2f})\nCapa {layer}')
            for item in ax:item.axis('off')
            fig.tight_layout();fig.savefig(output/f'{arch}_{frame["image_id"]}.png',dpi=160);plt.close(fig)
        del model;tf.keras.backend.clear_session()


## 9. Informe de procedimiento y resultados observados

In [ ]:
"""Informe exclusivamente con DATOS OBSERVADOS; no contiene metas numéricas de la tesis."""
from pathlib import Path
import pandas as pd



def check_report(df, results_dir):
    folder=Path(results_dir); folder.mkdir(parents=True,exist_ok=True)
    cols=['emotion_label','split'] if 'split' in df else ['emotion_label']
    counts=df.groupby(cols,dropna=False).size().rename('n').reset_index()
    lines=['# Control del procedimiento ejecutado','',
           'Este informe se calcula a partir del manifiesto proporcionado y los CSV generados por los modelos.',
           'No contiene porcentajes esperados, metas de exactitud ni predicciones precargadas.','',
           '## Conteos observados','','```csv\n'+counts.to_csv(index=False).strip()+'\n```','']
    f=folder/'metricas_finales_completas.csv'
    if f.exists():
        metrics=pd.read_csv(f)
        lines+=['## Métricas observadas en el test','','```csv\n'+metrics.to_csv(index=False).strip()+'\n```','',
                'Los resultados proceden de la ejecución de modelos con las imágenes disponibles.']
    else:
        lines+=['## Métricas','', 'No hay métricas de entrenamiento/evaluación en la carpeta. Ejecutar train y evaluate.']
    lines+=['','## Interpretación','',
            'La existencia del código no demuestra que se haya ejecutado en 2024 ni que replique los resultados publicados.',
            'Conservar registros originales, listas de imágenes, seeds, versiones de software y pesos de cada ejecución.']
    output=folder/'PROCEDIMIENTO_Y_RESULTADOS_OBSERVADOS.md'
    output.write_text('\n'.join(lines)+'\n',encoding='utf-8')
    return output


## 10. Ejecutar las etapas seleccionadas (en orden)
Estas celdas procesan exclusivamente los archivos proporcionados. Cambia las rutas y los indicadores de ejecución. Si tu CSV ya está preprocesado y etiquetado, deja desactivadas las dos primeras fases.

In [ ]:
# 10.1 Curación facial opcional: se ejecuta SOLO cuando se aportan capturas originales.
if EJECUTAR_CURACION_MTCNN:
    if not CSV_CAPTURAS_ORIGINALES:
        raise ValueError('Configurar CSV_CAPTURAS_ORIGINALES antes de curar rostros.')
    n_aceptadas,n_leidas=curate(CSV_CAPTURAS_ORIGINALES, str(CARPETA_TRABAJO/'curacion_privada'))
    print(f'Rostros aceptados {n_aceptadas}/{n_leidas} (datos observados)')
    CSV_MANIFIESTO_CURADO=str(CARPETA_TRABAJO/'curacion_privada'/'manifest_curado_privado.csv')
else:
    print('Curación MTCNN no ejecutada en esta sesión: se espera manifiesto previamente curado.')


In [ ]:
# 10.2 Sugerencias automáticas DeepFace+ViT: NO son verdades de terreno.
if EJECUTAR_SUGERENCIA_ETIQUETAS_DEEPFACE_VIT:
    if not EJECUTAR_CURACION_MTCNN:
        raise ValueError('Active curación o proporcione explícitamente CSV_MANIFIESTO_CURADO.')
    CSV_SUGERENCIAS=str(CARPETA_TRABAJO/'sugerencias_deepface_vit_PRIVADAS.csv')
    suggest_labels(CSV_MANIFIESTO_CURADO,CSV_SUGERENCIAS)
    print('Revise las sugerencias. No se convierten automáticamente en etiquetas humanas.')
if EJECUTAR_USO_EXPLICITO_DE_PSEUDOETIQUETAS:
    if not EJECUTAR_SUGERENCIA_ETIQUETAS_DEEPFACE_VIT:
        raise ValueError('No existen sugerencias recién generadas para integrar.')
    if not PERMITIR_PSEUDOETIQUETAS:
        raise ValueError('Para usar pseudoetiquetas deberá activar PERMITIR_PSEUDOETIQUETAS y reconocer su limitación.')
    salida=str(CARPETA_TRABAJO/'manifest_etiquetado_pseudo_PRIVADO.csv')
    print(assemble_suggestions(CSV_MANIFIESTO_CURADO,CSV_SUGERENCIAS,salida,allow_automatic_labels=True))
    CSV_MANIFIESTO_ETIQUETADO=salida
print('Archivo de etiquetas seleccionado:',CSV_MANIFIESTO_ETIQUETADO or '(sin seleccionar)')


In [ ]:
# 10.3 Auditar etiquetas, separar 80/20 y guardar el manifiesto del experimento.
if not CSV_MANIFIESTO_ETIQUETADO:
    raise ValueError('Debes indicar CSV_MANIFIESTO_ETIQUETADO: CSV privado con image_id,image_path,subject_id,emotion_label,label_source.')

df_entrada=load_manifest(CSV_MANIFIESTO_ETIQUETADO,allow_pseudo=PERMITIR_PSEUDOETIQUETAS)
df=split_train_test(df_entrada,seed=SEMILLA,test_fraction=FRACCION_TEST)
check_split(df)
if (df['label_source']!='human_verified').any():
    print('AVISO: existen pseudoetiquetas. Las métricas NO prueban reconocimiento de emociones reales verificadas.')
print('Método de separación:',df.attrs.get('split_method'))
display(summary(df))

# Se guarda el manifiesto con rutas relativas; el CSV es PRIVADO.
manifesto_destino=CARPETA_TRABAJO/'manifest_split_PRIVADO.csv'
df_guardar=df.copy()
df_guardar['image_path']=[os.path.relpath(p,manifesto_destino.parent) for p in df_guardar['resolved_path']]
df_guardar.drop(columns=['resolved_path'],errors='ignore').to_csv(manifesto_destino,index=False)
print('Manifiesto con partición real:',manifesto_destino)
print('Huella del conjunto:',manifest_fingerprint(df))
write_environment(CARPETA_RESULTADOS,config,{'manifest_fingerprint':manifest_fingerprint(df),'labeling_policy':'pseudo' if PERMITIR_PSEUDOETIQUETAS else 'human_verified'})


In [ ]:
# 10.4 Tuning bayesiano real: 3 ensayos de 5 épocas por arquitectura, TRAIN exclusivamente.
# Los hiperparámetros surgen de la búsqueda; NO están cargados desde una tabla del informe.
if EJECUTAR_OPTIMIZACION_BAYESIANA:
    print('Optimizando las cinco arquitecturas con el conjunto de entrenamiento...')
    optimizados=tune_architectures(df, CARPETA_RESULTADOS/'tuning', config, max_trials=NUMERO_ENSAYOS_BAYESIANOS, trial_epochs=NUMERO_EPOCAS_TUNER)
    display(pd.DataFrame(optimizados).T)
else:
    print('Búsqueda bayesiana desactivada. Se necesita archivo real hiperparametros_optimizados.json para entrenar.')

ruta_hps=CARPETA_RESULTADOS/'tuning'/'hiperparametros_optimizados.json'
if (EJECUTAR_ENTRENAMIENTO_5_CNN or EJECUTAR_VALIDACION_CRUZADA_REAL_5FOLD) and not ruta_hps.exists():
    raise FileNotFoundError(f'No hay hiperparámetros de una búsqueda ejecutada: {ruta_hps}')
hiperparametros=load_hyperparameters(str(ruta_hps)) if ruta_hps.exists() else None


In [ ]:
# 10.5 Entrenamiento final (cinco modelos). Almacenamiento privado.
if EJECUTAR_ENTRENAMIENTO_5_CNN:
    resultados_entrenamiento=train_final(df,CARPETA_RESULTADOS,config,hiperparametros)
    display(pd.DataFrame(resultados_entrenamiento))
else:
    print('Entrenamiento final no ejecutado en esta sesión.')


In [ ]:
# 10.6 Validación cruzada auténtica OPCIONAL (5 folds del conjunto train).
# IMPORTANTÍSIMO: reentrena 5 modelos por arquitectura; NO debe afirmarse que se hizo si la celda no se ejecutó.
if EJECUTAR_VALIDACION_CRUZADA_REAL_5FOLD:
    cv_resultados=crossvalidate(df,CARPETA_RESULTADOS,config,hiperparametros)
    display(cv_resultados)
else:
    print('Validación cruzada con reentrenamiento NO ejecutada. En el código original solo había 5 subconjuntos del test.')


In [ ]:
# 10.7 Evaluación FINAL de test independiente y matriz de confusión de cada CNN.
if EJECUTAR_EVALUACION_TEST:
    tabla_final=evaluate_models(df,CARPETA_RESULTADOS,batch_size=config.batch_size,seed=SEMILLA)
    display(tabla_final)
    print('Se creó estabilidad_5_subconjuntos_TEST_NO_CV.csv (NO es reentrenamiento 5-fold).')
else:
    print('Evaluación final no ejecutada.')


In [ ]:
# 10.8 Robustez: control, gafas, mascarilla, sombras, ángulo y estrés compuesto.
if EJECUTAR_PRUEBAS_ROBUSTEZ:
    tabla_estres=run_stress(df,CARPETA_RESULTADOS,batch_size=config.batch_size,seed=SEMILLA)
    display(tabla_estres)
else:
    print('Pruebas de robustez no ejecutadas.')


In [ ]:
# 10.9 Grad-CAM — exporta imágenes faciales PRIVADAS únicamente si se autoriza.
if EJECUTAR_GRADCAM_PRIVADO:
    export_gradcam(df,CARPETA_RESULTADOS,n_per_class=1,allow_private_export=True)
    print('Mapas privados generados, NO PUBLICAR EN GITHUB.')
else:
    print('Grad-CAM no ejecutado; requiere autorización expresa para generar imágenes de participantes.')


In [ ]:
# 10.10 Informe final: únicamente métricas calculadas, sin porcentajes objetivo.
informe=check_report(df,CARPETA_RESULTADOS)
print('Informe reproducible observado:',informe)
print('Se almacenan en carpeta privada los modelos, CSV y gráficos de la ejecución.')
print('Antes de publicar este notebook en GitHub, use EDITAR > BORRAR TODAS LAS SALIDAS.')


---
## Alcance de la evidencia

- El código del experimento recibido incorporaba **MTCNN, DeepFace + ViT, CLAHE, 80/20, cinco CNN, sintonización, fine-tuning, métricas, Grad-CAM y robustez**. Este cuaderno organiza esas etapas con controles de calidad y sin filtración del test.
- El texto recibido evaluaba **cinco subconjuntos del test con pesos fijos**; el CSV de estabilidad lo denomina así, no «cross-validation». La celda independiente de **5-fold real** está disponible, pero solo se convierte en evidencia cuando se ejecuta con datos originales y guarda sus métricas.
- En el código original no se verificaron experimentalmente técnicas como `label_smoothing=0.05` o `ReduceLROnPlateau`: por fidelidad a la fuente, quedan **desactivadas** en la configuración base. Si se usan en una ejecución nueva, declárese expresamente.
- Las métricas obtenidas con pseudoetiquetas deben describirse como concordancia con dichas etiquetas, no como verdad emocional confirmada. No existe garantía de reproducir porcentajes antiguos si se cambia el protocolo.
- Para GitHub, publique una **copia sin datos y sin salidas**, con las rutas privadas reemplazadas por valores de ejemplo.
